<a href="https://colab.research.google.com/github/CPTR295/Learning-Langchain/blob/main/Agents_Basic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain_openai langchain_community

In [ ]:
!pip install -U ddgs

In [1]:
from google.colab import userdata
api_key = userdata.get("OPENROUTER_API_KEY")

In [12]:
from typing import Annotated, TypedDict
import ast
from langchain_openai import ChatOpenAI
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.graph import START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode,tools_condition

@tool
def calculator(query: str) -> str:
  """A simple calculator tool. Input should be a mathematical expression.""" #Every tools needs a docstring like a description
  return ast.literal_eval(query)

from sympy import sympify

@tool
def calculator(query: str):
  """A simple calculator tool. Input should be a mathematical expression.""" #Every tools needs a docstring like a description
  try:
      result = sympify(query)

      if not result.is_number:
          return "Invalid mathematical expression"

      return str(result)

  except Exception as e:
      return f"Calculation error: {e}"

search = DuckDuckGoSearchRun()
tools = [search,calculator]
model = ChatOpenAI( #Many providers provide same api interface as openai including openrouter
    model="nvidia/nemotron-3-ultra-550b-a55b:free",
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1",
    temperature=0.1).bind_tools(tools)

class State(TypedDict):
    messages: Annotated[list, add_messages]

def model_nodes(state:State) -> State:
  res = model.invoke(state['messages'])
  return {"messages":res}

builder = StateGraph(State)
builder.add_node('model',model_nodes)
builder.add_node('tools',ToolNode(tools))
builder.add_edge(START,'model')
builder.add_conditional_edges('model',tools_condition)
builder.add_edge("tools",'model')

graph = builder.compile()

imput = {
    "messages" :[HumanMessage("How old was the 30th president of the United States when he died ?")]
}

for c in graph.stream(imput):
  print(c)

{'model': {'messages': AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 179, 'prompt_tokens': 386, 'total_tokens': 565, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 138, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'nvidia/nemotron-3-ultra-550b-a55b:free', 'system_fingerprint': None, 'id': 'gen-1793443331-1TTjIyHhGCy9I05d2nUe', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1918d-7c26-704d-beb9-8c950ed54908-0', tool_calls=[{'name': 'duckduckgo_search', 'args': {'query': '30th president of United States age at death'}, 'id': 'call-

In [16]:
#TO force first function
from langchain_core.messages import AIMessage, HumanMessage, ToolCall
from uuid import uuid4
def first_model(state: State) -> State:
    query = state["messages"][-1].content
    search_tool_call = ToolCall(
        name="duckduckgo_search", args={"query": query}, id=uuid4().hex
    )
    return {"messages": AIMessage(content="", tool_calls=[search_tool_call])}


builder = StateGraph(State)
builder.add_node("first_model", first_model)
builder.add_node("model", model_nodes)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "first_model")
builder.add_edge("first_model", "tools")
builder.add_conditional_edges("model", tools_condition)
builder.add_edge("tools", "model")

graph = builder.compile()

# Example usage
input = {
    "messages": [
        HumanMessage(
            "How old was the 30th president of the United States when he died?"
        )
    ]
}

for c in graph.stream(input):
    print(c)

{'first_model': {'messages': AIMessage(content='', additional_kwargs={}, response_metadata={}, id='09f579d4-fff9-4eb0-9e3e-d51661c806cb', tool_calls=[{'name': 'duckduckgo_search', 'args': {'query': 'How old was the 30th president of the United States when he died?'}, 'id': '2df0917c2bf24e63bedeaae5f10d7bc3', 'type': 'tool_call'}], invalid_tool_calls=[])}}
{'tools': {'messages': [ToolMessage(content='List of presidents of the United States The White House \'s north and south sides, official residence of the president of the United States The president of the United States is the head of state and head of government of the United States, [1] indirectly elected to a four-year term via the Electoral College. [2] His grandson, Benjamin Harrison, became the 23rd president of the United States. In 1794, he participated in the Battle of Fallen Timbers, an American military victory that ended the Northwest Indian War. In 1811, he led a military force against Tecumseh\'s confederacy at the Battl

In [17]:

import requests
from langchain_core.embeddings import Embeddings
from google.colab import userdata

api_key = userdata.get("OPENROUTER_API_KEY")

class OpenRouterNvidiaEmbeddings(Embeddings):
    def __init__(self, api_key):
        self.api_key = api_key
        self.url = "https://openrouter.ai/api/v1/embeddings"
        self.model = "nvidia/llama-nemotron-embed-vl-1b-v2:free"

    def _embed(self, texts, input_type):
        response = requests.post(
            self.url,
            headers={
                "Authorization": f"Bearer {self.api_key}",
                "Content-Type": "application/json",
            },
            json={
                "model": self.model,
                "input": [
                    {
                        "content": [
                            {"type": "text", "text": text}
                        ]
                    }
                    for text in texts
                ],
                "input_type": input_type,
                "encoding_format": "float",
            },
            timeout=120,
        )

        response.raise_for_status()
        data = response.json()["data"]
        data.sort(key=lambda item: item["index"])

        return [item["embedding"] for item in data]

    def embed_documents(self, texts):
        return self._embed(texts, "passage")

    def embed_query(self, text):
        return self._embed([text], "query")[0]


embeddings_model = OpenRouterNvidiaEmbeddings(api_key)

In [19]:
import ast
from typing import Annotated, TypedDict

from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_core.vectorstores.in_memory import InMemoryVectorStore
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from langgraph.graph import START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition


search = DuckDuckGoSearchRun()
tools = [search, calculator]

embeddings = embeddings_model


tools_retriever = InMemoryVectorStore.from_documents(
    [Document(tool.description, metadata={"name": tool.name}) for tool in tools],
    embeddings,
).as_retriever()


class State(TypedDict):
    messages: Annotated[list, add_messages]
    selected_tools: list[str]


def model_node(state: State) -> State:
    selected_tools = [tool for tool in tools if tool.name in state["selected_tools"]]
    res = model.bind_tools(selected_tools).invoke(state["messages"])
    return {"messages": res}


def select_tools(state: State) -> State:
    query = state["messages"][-1].content
    tool_docs = tools_retriever.invoke(query)
    return {"selected_tools": [doc.metadata["name"] for doc in tool_docs]}


builder = StateGraph(State)
builder.add_node("select_tools", select_tools)
builder.add_node("model", model_node)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "select_tools")
builder.add_edge("select_tools", "model")
builder.add_conditional_edges("model", tools_condition)
builder.add_edge("tools", "model")

graph = builder.compile()

# Example usage
input = {
    "messages": [
        HumanMessage(
            "How old was the 30th president of the United States when he died?"
        )
    ]
}

for c in graph.stream(input):
    print(c)

{'select_tools': {'selected_tools': ['calculator', 'duckduckgo_search']}}
{'model': {'messages': AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 129, 'prompt_tokens': 386, 'total_tokens': 515, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 87, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'nvidia/nemotron-3-ultra-550b-a55b:free', 'system_fingerprint': None, 'id': 'gen-1793443710-54BtjbEwliTtVeMmeEGN', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a19193-4647-7795-b20e-5b8ad3974af2-0', tool_calls=[{'name': 'duckduckgo_search', 'args'